In [0]:
from pyspark.sql import functions as F

In [0]:
CATALOG = "clinical_trial_supply_chain"

SILVER = f"{CATALOG}.silver"
GOLD = f"{CATALOG}.gold"

INVENTORY = f"{SILVER}.fact_inventory"
SITE = f"{SILVER}.dim_site_scd2"
COUNTRY = f"{SILVER}.dim_country"

TARGET = f"{GOLD}.gold_country_inventory"

In [0]:
site_df = (
    spark.table(SITE)
         .filter("is_current = true")
         .select(
             "site_id",
             "country_code"
         )
)

In [0]:
site_df = (
    spark.table(SITE)
         .filter("is_current = true")
         .select(
             "site_id",
             "country_code"
         )
)

In [0]:
country_inventory_df = (
    spark.table(INVENTORY).alias("inv")
    .join(
        site_df.alias("site"),
        "site_id",
        "inner"
    )
    .join(
        spark.table(COUNTRY).alias("country"),
        "country_code",
        "left"
    )
    .groupBy(
        "country_code",
        "country_name",
        "region",
        "drug_type"
    )
    .agg(
        F.sum("qty_on_hand")
            .alias("total_on_hand"),

        F.sum("available_stock")
            .alias("total_available_stock"),

        F.sum("qty_reserved")
            .alias("total_reserved_stock"),

        F.sum("qty_expired")
            .alias("total_expired_stock")
    )
)

In [0]:
(
    country_inventory_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TARGET)
)

In [0]:
display(
    spark.table(TARGET)
)